<a href="https://colab.research.google.com/github/syedmahmoodiagents/genai_classes/blob/main/StateGraph_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install langchain-huggingface --q

In [9]:
!pip install langgraph --q

In [10]:
from typing import TypedDict, Literal

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver

In [11]:
import os

In [12]:
os.environ["HF_TOKEN"] = "hf_zIvXpRihcLDYQxQSByHOzDVeLxVhaQMEg"

In [13]:
llm = ChatHuggingFace(llm = HuggingFaceEndpoint(repo_id="openai/gpt-oss-20b"))

In [14]:
class State(TypedDict):
    question: str
    operation: str
    a: int
    b: int
    answer: int


In [15]:
def addition(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


def multiplication(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


In [ ]:
def addition_node(state: State):
    result = addition(state["a"], state["b"])

    return {"answer": result}


def multiplication_node(state: State):
    result = multiplication(state["a"], state["b"])

    return {"answer": result}


In [16]:
def agent(state: State):

    question = state["question"]

    response = llm.invoke(
        f"""
        You are a mathematical routing agent.

        Determine which operation the user wants.

        User question:
        {question}

        Respond with ONLY one of these words:

        ADD
        MULTIPLY
        """
    )

    operation = response.content.strip().upper()

    import re
    numbers = re.findall(r"-?\d+", question)

    # if len(numbers) != 2:
    #     raise ValueError(
    #         "Question must contain exactly two numbers."
    #     )

    a = int(numbers[0])
    b = int(numbers[1])

    return {"operation": operation, "a": a, "b": b}


In [ ]:


# Router

def route_operation(state: State):

    if state["operation"] == "ADD":
        return "addition"

    elif state["operation"] == "MULTIPLY":
        return "multiplication"



builder = StateGraph(State)
builder.add_node("agent", agent)
builder.add_node("addition", addition_node)
builder.add_node("multiplication", multiplication_node)

builder.set_entry_point("agent")
# Conditional routing
builder.add_conditional_edges("agent", route_operation, { "addition": "addition", "multiplication": "multiplication"})

builder.add_edge("addition", END)
builder.add_edge("multiplication", END)




In [ ]:
agent_pipeline = builder.compile(checkpointer=InMemorySaver())

In [ ]:
config = {"configurable": {"thread_id": "user-1"}}
result = agent_pipeline.invoke({"question": "What is 25 plus 15?"},config=config)

print("Result 1:")
print(result)

result = agent_pipeline.invoke({"question": "What is 25 multiplied by 15?"},config=config)
print("\nResult 2:")
print(result)

In [7]:
import re
re.findall(r"-?\d+", "there are 70 fruits and 40 vegetables")

['70', '40']